## Use RAG with API， google search and the local document retrieval 

用户问题
   ↓
本地文档检索 retrieval 
   ↓
找出最相关的本地 chunks
   ↓
Google Search 检索， tools = Google Search
拿到实时网页结果
   ↓
把
- question
- local retrieval context
- google search context
一起构造成最终 prompt
   ↓
Gemini 回答


Google Search 作为 Gemini 的内置工具，本地文档作为你拼进去的 context or 你先自己拿到 Google 搜索摘要，再和本地文档一起拼进 prompt 


 

In [ ]:
! pip install google-genai chromadb sentence-transformers pypdf  

In [ ]:
# 
import os
import glob
# To give the unique id for the chromadb 
import uuid
from typing import List, Dict

import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader 

In [ ]:
#Gloabal vairable name 
data_dir = "data" 
chroma_collection_name = 'local_docs' 
embeded_model_name =  "all-MiniLM-L6-v2" 

In [ ]:
# Function to read the text and the pdf file 
def read_txt_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def read_pdf_file(path: str) -> str:
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(f"[Page {i+1}]\n{text}")
    return "\n\n".join(pages) 
 

In [ ]:
# Put the txt and the pdf file  text into the doc list
# the list should be [ { source: ... , "text": ....}]
def load_documents(data_dir: str) -> List[Dict]:
    docs = []

    for path in glob.glob(os.path.join(data_dir, "*.txt")):
        text = read_txt_file(path)
        docs.append({
            "source": os.path.basename(path),
            "text": text
        })

    for path in glob.glob(os.path.join(data_dir, "*.pdf")):
        text = read_pdf_file(path)
        docs.append({
            "source": os.path.basename(path),
            "text": text
        })

    return docs 



In [ ]:
# The chromadb can't store the whole text, need to chunk 
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
    """
    一个简单字符级 chunker。
    真实项目里你也可以换成 token-based chunking。
    """

    chunks = []
    start = 0
    text = text.strip()

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks



In [ ]:
# The chromadb can't store the whole text, need to chunk 
def chunk_embed_text(text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
    """
    真实项目里你也可以换成 token-based chunking。
    """
    #The input is not the text character but the embedded text. 
    
    


In [ ]:




def build_index( data_dir, embed_model_name, chroma_collection_name ):
    
    
    
    docs = load_documents(data_dir)
    print(f"Loaded {len(docs)} documents.")

    embed_model = SentenceTransformer( embed_model_name)

    # the chromadb.PersistentClient
    try :  
       client =  chromadb.Client(path = "./chroma_db_1")
    except:  
        client = chromadb.PersistentClient(path="./chroma_db_2")

    # 如果集合已存在，可先删除再重建
    existing = [c.name for c in client.list_collections()]
    if  chroma_collection_name  in existing:
        client.delete_collection( chroma_collection_name  )

    collection = client.create_collection(name= chroma_collection_name  )

    all_ids = []
    all_texts = []
    all_embeddings = []
    all_metadatas = []

    for doc in docs:
        chunks = chunk_text(doc["text"])
        embeddings = embed_model.encode(chunks).tolist()

        for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
            all_ids.append(str(uuid.uuid4())) # add the index uuid.uuid4() to trans to string 
            all_texts.append(chunk)
            all_embeddings.append(emb)
            all_metadatas.append({
                "source": doc["source"],
                "chunk_id": i
            })

    collection.add(
        ids=all_ids,
        documents=all_texts,
        embeddings=all_embeddings,
        metadatas=all_metadatas
    )

    print(f"Indexed {len(all_texts)} chunks into ChromaDB.")
 

## Local retrieval and the Google search tool 

In [ ]:
import os
from typing import List, Dict

import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types

# 你需要先设置环境变量：
# export GEMINI_API_KEY="your_api_key"
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")


def retrieve_local_context(query: str, top_k: int = 4) -> List[Dict]:
    embed_model = SentenceTransformer(embeded_model_name)
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_collection(chroma_collection_name)

    #get the retreival result with the chroma collection.query(query_ebedding, n_results )
    query_embedding = embed_model.encode([query])[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results.get("distances", [[]])[0]

    retrieved = []
    
    for doc, meta, dist in zip(docs, metas, distances if distances else [None] * len(docs)):
        retrieved.append({
            "text": doc,
            "source": meta.get("source"),
            "chunk_id": meta.get("chunk_id"),
            "distance": dist
        })

    return retrieved


def format_local_context(local_docs: List[Dict]) -> str:
    blocks = []
    for i, item in enumerate(local_docs, start=1):
        blocks.append(
            f"[LOCAL_DOC_{i}] "
            f"source={item['source']} chunk_id={item['chunk_id']}\n"
            f"{item['text']}"
        )
    return "\n\n".join(blocks)


def build_final_prompt(question: str, local_context: str) -> str:
    """
    这里本地文档显式拼进 prompt。
    Google Search 不需要你手动去搜网页，
    Gemini 会通过 google_search tool 自动补充网页信息。
    """
    prompt = f"""
    You are a retrieval-augmented question answering assistant.

    You must answer the user's question by combining:
    1. Local retrieved documents provided below
    2. Relevant real-time web information from Google Search grounding
    3. The user's question

    Instructions:
    - Prefer local documents when they directly answer the question.
    - Use Google Search grounding for recent, dynamic, or missing information.
    - If local documents and web results disagree, explicitly mention the discrepancy.
    - Cite which parts come from local documents vs web grounding.
    - Do not invent facts not supported by either local context or web results.
    - If the answer is uncertain, say so clearly.

    ====================
    LOCAL RETRIEVED CONTEXT
    ====================
    {local_context}

    ====================
    USER QUESTION
    ====================
    {question}

    Now provide:
    1. A concise answer
    2. A detailed explanation
    3. A short section called "Sources used" separating:
    - Local documents
    - Google Search grounded results
    """
    return prompt.strip()


def answer_with_gemini(question: str):
    local_docs = retrieve_local_context(question, top_k=4)
    local_context = format_local_context(local_docs)
    final_prompt = build_final_prompt(question, local_context)

    client = genai.Client(api_key=GEMINI_API_KEY)
    # Gemini to call the function 
    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=final_prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())], # the tools is emebeded inside of the Geminit 
            
            temperature=0.2
        )
    )

    return {
        "local_docs": local_docs,
        "prompt": final_prompt,
        "answer_text": response.text,
        "raw_response": response
    }



In [ ]:

if __name__ == "__main__":
    #Get the retrieval chromadb storage for the vector in the chunk size. 
    build_index() 
    
    # 
    question = "Based on my local documents and recent web information, what are the latest best practices for building a RAG system?"
    result = answer_with_gemini(question)

    print("=" * 80)
    print("FINAL PROMPT")
    print("=" * 80)
    print(result["prompt"])

    print("\n" + "=" * 80)
    print("ANSWER")
    print("=" * 80)
    print(result["answer_text"])

    print("\n" + "=" * 80)
    print("LOCAL RETRIEVAL")
    print("=" * 80)
    for item in result["local_docs"]:
        print(item["source"], item["chunk_id"])  

## Not use the internal googlesearch，but with tool-calling / agent 
That means 搜索是你自己发 HTTP 请求， 自己解析， rompt 是你自己拼接， 模型并不知道“Google Search”是个神秘内置能力， 对模型来说，它只看到了你喂进去的文本上下文 


User Question
   ↓
Local Retriever
   ↓
Local top-k chunks

User Question
   ↓
Search Tool (Google search API / SerpAPI / Custom Search)
   ↓
Web results

[Question + Local Context + Web Context]
   ↓
Local LLM
   ↓
Final answer

Google Custom Search JSON API： REST API 以 JSON 返回搜索结果；
使用前通常需要先创建 Programmable Search Engine 并拿到 API key。

In [ ]:
############ web 
GOOGLE_API_KEY=os.environ["your_google_api_key"]
GOOGLE_CSE_ID=os.environ["your_programmable_search_engine_id" ]
import request 
def google_search_tool(query: str, num_results: int = 5) -> List[Dict]:
    """
    显式调用 Google Custom Search JSON API
    返回结构化搜索结果
    """
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": GOOGLE_API_KEY,
        "cx": GOOGLE_CSE_ID,
        "q": query,
        "num": num_results,
    }

    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    results = []
    for item in data.get("items", []):
        results.append({
            "title": item.get("title", ""),
            "link": item.get("link", ""),
            "snippet": item.get("snippet", ""),
            "displayLink": item.get("displayLink", "")
        })
    return results



########## Local 

import chromadb
from sentence_transformers import SentenceTransformer
from typing import List, Dict

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
CHROMA_COLLECTION = "local_docs"


def retrieve_local_context(query: str, top_k: int = 4) -> List[Dict]:
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_collection(CHROMA_COLLECTION)

    query_embedding = embed_model.encode([query])[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    output = []
    for doc, meta in zip(docs, metas):
        output.append({
            "text": doc,
            "source": meta.get("source", "unknown"),
            "chunk_id": meta.get("chunk_id", -1)
        })
    return output


################# 

def format_local_context(local_docs: List[Dict]) -> str:
    blocks = []
    for i, doc in enumerate(local_docs, start=1):
        blocks.append(
            f"[LOCAL_{i}] source={doc['source']} chunk_id={doc['chunk_id']}\n"
            f"{doc['text']}"
        )
    return "\n\n".join(blocks)

def format_web_context(web_results: List[Dict]) -> str:
    blocks = []
    for i, item in enumerate(web_results, start=1):
        blocks.append(
            f"[WEB_{i}] title={item['title']}\n"
            f"url={item['link']}\n"
            f"snippet={item['snippet']}"
        )
    return "\n\n".join(blocks)


#################

def build_final_prompt(question: str, local_context: str, web_context: str) -> str:
    return f"""
You are a question-answering assistant.

You must answer the user's question using:
1. Local retrieved documents
2. Google web search results

Rules:
- Prefer local documents for user-specific or internal facts.
- Prefer web results for recent or changing facts.
- If the two sources disagree, explicitly mention the discrepancy.
- Do not invent unsupported claims.
- Clearly separate which evidence comes from local docs vs web results.

====================
LOCAL RETRIEVAL
====================
{local_context}

====================
GOOGLE SEARCH RESULTS
====================
{web_context}

====================
QUESTION
====================
{question}

Return:
1. Short answer
2. Detailed explanation
3. Evidence summary
""".strip() 



from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # 举例


def load_local_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )
    return tokenizer, model


def generate_answer(prompt: str, tokenizer, model, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": "You are a careful RAG assistant."},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return answer 

In [ ]:
def rag_with_explicit_google_search(question: str, tokenizer, model):
    # 1) local retrieval
    local_docs = retrieve_local_context(question, top_k=4)
    local_context = format_local_context(local_docs)

    # 2) web search
    web_results = google_search_tool(question, num_results=5)
    web_context = format_web_context(web_results)

    # 3) final prompt
    final_prompt = build_final_prompt(question, local_context, web_context)

    # 4) local llm answer
    answer = generate_answer(final_prompt, tokenizer, model)

    return {
        "question": question,
        "local_docs": local_docs,
        "web_results": web_results,
        "prompt": final_prompt,
        "answer": answer
    }


if __name__ == "__main__":
    tokenizer, model = load_local_model()

    question = "What are the latest best practices for building a RAG system?"
    result = rag_with_explicit_google_search(question, tokenizer, model)

    print("=" * 80)
    print("FINAL PROMPT")
    print("=" * 80)
    print(result["prompt"])

    print("\n" + "=" * 80)
    print("ANSWER")
    print("=" * 80)
    print(result["answer"]) 